# AnyCompany Consulting - 고객 지원 이메일 에이전트 (LangGraph + Amazon Nova)

고객 지원 부서는 시간당 1,000통 이상의 이메일을 받습니다. 이 노트북은 다음 일을 하는 이메일 에이전트를 **LangGraph**와 **Amazon Bedrock의 Nova 모델**로 구현합니다.

1. 스팸 메일과 실제 이메일 분류
2. 간단한 질문에 자동 응답
3. 긴급하거나 복잡한 문제는 사람에게 에스컬레이션
4. 진행 중인 스레드의 후속 이메일 추적·관리

## 워크플로 패턴: 라우팅(Routing) + 스레드 상태 저장 + Human-in-the-loop

```
START → load_thread → classify ─┬─ spam     → mark_spam ─────────────────────────────→ END
                                ├─ simple   → auto_reply ────────────────────────────→ END
                                │                 └(FAQ로 답할 수 없음)┐
                                └─ escalate → escalate(티켓) → human_review(interrupt) → send_human_reply → END
```

**이 패턴의 이점**
- **비용과 속도:** 대부분을 차지하는 스팸·단순 질문은 가벼운 모델 호출 한두 번으로 끝나고, 사람의 시간은 꼭 필요한 메일에만 씁니다.
- **역할 분리:** 경로마다 노드와 프롬프트가 따로 있어 한 경로를 고쳐도 다른 경로에 영향이 없습니다.
- **예측 가능성:** 흐름을 코드로 정해두어 동작을 추적·감사하기 쉽습니다.
- **안전장치:** 확신이 없으면 사람에게 넘기고, 중요한 답변은 사람이 확인한 뒤 나갑니다.
- **상태 유지:** 같은 스레드의 후속 메일이 같은 티켓과 맥락에 연결됩니다.

## 1. 준비

- AWS 자격 증명이 설정되어 있어야 합니다 (`aws configure` 또는 `aws sso login`).
- Bedrock 콘솔에서 사용할 Nova 모델의 **모델 액세스**가 켜져 있어야 합니다.
- IAM 권한: `bedrock:InvokeModel` (Converse API 호출에 필요)

In [ ]:
%pip install -q langgraph boto3 pydantic grandalf

## 2. 설정

기본 모델은 **Nova 2 Lite**입니다. Converse API는 모델 ID만 바꾸면 다른 Nova 모델도 그대로 쓸 수 있습니다.

| 모델 | 교차 리전 추론 프로필 ID (US) | 특징 |
|---|---|---|
| Nova Micro | `us.amazon.nova-micro-v1:0` | 텍스트 전용, 가장 저렴·빠름 |
| Nova Lite | `us.amazon.nova-lite-v1:0` | 저렴한 멀티모달 |
| Nova 2 Lite | `us.amazon.nova-2-lite-v1:0` | 추론 능력이 향상된 2세대 (기본값) |
| Nova Pro | `us.amazon.nova-pro-v1:0` | 복잡한 작업용 고성능 |

`us.` 접두사는 미국 리전 간 교차 리전 추론 프로필입니다. 다른 지역이면 `eu.`/`apac.` 프로필을 쓰세요.

In [ ]:
import os

AWS_REGION = os.getenv("AWS_REGION", "us-west-2")
MODEL_ID = os.getenv("NOVA_MODEL_ID", "us.amazon.nova-2-lite-v1:0")

print(f"Region: {AWS_REGION}, Model: {MODEL_ID}")

## 3. 라이브러리와 Bedrock 클라이언트

In [ ]:
import operator
import uuid
from typing import Annotated, Literal, Optional, TypedDict

import boto3
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel, Field

bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)

## 4. 지식 베이스(FAQ)

간단한 질문에 답할 때 참고하는 정보입니다. 실서비스에서는 Amazon Bedrock Knowledge Bases 같은 RAG로 바꾸면 됩니다.

In [ ]:
FAQ = """
- 영업시간: 평일 09:00~18:00 (KST), 주말/공휴일 휴무
- 비밀번호 재설정: 로그인 화면의 '비밀번호 찾기' 클릭 후 가입 이메일로 링크 수신
- 청구서 확인: 대시보드 > 결제 > 청구서 메뉴에서 PDF 다운로드
- 요금제 변경: 대시보드 > 구독 관리에서 언제든 변경 가능, 다음 결제일부터 적용
- 컨설팅 미팅 예약: https://anycompany.example.com/booking
"""

## 5. 상태와 출력 스키마

- `EmailState`: 그래프가 주고받는 상태입니다. `history`는 reducer(`operator.add`)로 누적되고, checkpointer가 `thread_id`별로 보관합니다.
- `Classification`, `ReplyDraft`: LLM이 반드시 이 형식으로 답하도록 강제할 Pydantic 스키마입니다.

In [ ]:
class Email(TypedDict):
    sender: str
    subject: str
    body: str


class Classification(BaseModel):
    """이메일 분류 결과"""

    route: Literal["spam", "simple", "escalate"] = Field(
        description="spam: 스팸/피싱/광고, simple: FAQ로 답변 가능한 단순 질문, "
        "escalate: 긴급하거나 복잡해서 사람이 처리해야 하는 문제"
    )
    urgency: Literal["low", "medium", "high"] = Field(description="긴급도")
    summary: str = Field(description="이메일 내용 한 줄 요약 (한국어)")
    reason: str = Field(description="이 경로를 선택한 이유 (한국어, 한 문장)")


class ReplyDraft(BaseModel):
    """고객에게 보낼 답장 초안"""

    reply: str = Field(description="고객에게 보낼 한국어 답장 본문")
    answered_from_faq: bool = Field(
        description="FAQ 정보만으로 답변이 충분하면 true, 아니면 false"
    )


class EmailState(TypedDict, total=False):
    # 이번에 수신한 이메일 (매 호출마다 입력)
    email: Email
    # 스레드 전체 이력 - 누적되며 checkpointer가 thread_id별로 보관
    history: Annotated[list[dict], operator.add]
    # 스레드 수준 상태 (후속 메일 추적용)
    is_followup: bool
    ticket_id: Optional[str]
    # 이번 메일 처리 결과
    classification: dict
    outcome: str
    reply: Optional[str]

## 6. Nova 호출 헬퍼 (구조화 출력)

Bedrock **Converse API**의 도구 호출(tool use)로 구조화 출력을 받습니다.

1. Pydantic 스키마를 도구의 `inputSchema`로 등록하고,
2. `toolChoice`로 그 도구를 **반드시 호출**하게 한 뒤,
3. 모델이 넘긴 도구 입력(JSON)을 Pydantic으로 검증합니다.

분류 작업이므로 `temperature=0`으로 결과를 일정하게 유지합니다.

In [ ]:
def llm_parse(system: str, user: str, schema: type[BaseModel]) -> BaseModel:
    tool_name = schema.__name__
    response = bedrock.converse(
        modelId=MODEL_ID,
        system=[{"text": system}],
        messages=[{"role": "user", "content": [{"text": user}]}],
        toolConfig={
            "tools": [{
                "toolSpec": {
                    "name": tool_name,
                    "description": schema.__doc__ or tool_name,
                    "inputSchema": {"json": schema.model_json_schema()},
                }
            }],
            "toolChoice": {"tool": {"name": tool_name}},
        },
        inferenceConfig={"maxTokens": 2048, "temperature": 0},
    )
    for block in response["output"]["message"]["content"]:
        if "toolUse" in block:
            return schema.model_validate(block["toolUse"]["input"])
    raise RuntimeError(f"구조화 출력을 받지 못했습니다: stopReason={response['stopReason']}")


def format_history(history: list[dict]) -> str:
    if not history:
        return "(이전 대화 없음)"
    return "\n".join(f"[{h['role']}] {h['content']}" for h in history[-10:])


def indent(text: str) -> str:
    return "\n".join("      │ " + line for line in text.splitlines())


# 동작 확인
llm_parse("이메일을 분류하세요.", "축하합니다! 아이폰에 당첨되셨습니다. 카드 번호를 입력하세요.", Classification)

## 7. 노드 정의

| 노드 | 역할 |
|---|---|
| `load_thread` | 스레드 이력으로 후속 메일인지 판단하고, 수신 메일을 이력에 추가 |
| `classify` | Nova로 spam / simple / escalate 분류 |
| `mark_spam` | 스팸 처리 |
| `auto_reply` | FAQ로 자동 답장. 답할 수 없으면 `Command(goto="escalate")` |
| `escalate` | 티켓 생성, 또는 기존 티켓에 후속 메일 연결 |
| `human_review` | `interrupt()`로 멈추고 담당자 답변 대기 |
| `send_human_reply` | 담당자 답장 발송·이력 기록 |

In [ ]:
def load_thread(state: EmailState) -> dict:
    email = state["email"]
    is_followup = len(state.get("history", [])) > 0
    followup = ""
    if is_followup:
        followup = "  (후속 메일" + (f", 티켓 {state['ticket_id']}" if state.get("ticket_id") else "") + ")"
    print(f"\n📨 수신: [{email['subject']}] from {email['sender']}{followup}")
    return {
        "is_followup": is_followup,
        "history": [{"role": "customer", "content": f"{email['subject']} - {email['body']}"}],
        # 이전 메일의 처리 결과는 초기화
        "classification": {},
        "outcome": "",
        "reply": None,
    }


def classify(state: EmailState) -> dict:
    email = state["email"]
    system = (
        "당신은 AnyCompany Consulting 고객 지원팀의 이메일 분류기입니다. "
        "수신 이메일을 spam / simple / escalate 중 하나로 분류하세요.\n"
        "- 아래 FAQ로 답할 수 있는 단순 질문은 simple\n"
        "- 서비스 장애, 데이터 유실, 보안, 환불/계약 분쟁, 강한 불만, FAQ로 답할 수 없는 문제는 escalate\n"
        "- 이미 담당자에게 에스컬레이션된 스레드(티켓 존재)의 후속 메일은 스팸이 아닌 한 escalate\n"
        "- summary와 reason은 한국어로 작성\n"
        f"\nFAQ:\n{FAQ}"
    )
    user = (
        f"스레드 티켓: {state.get('ticket_id') or '없음'}\n"
        f"이전 대화:\n{format_history(state.get('history', [])[:-1])}\n\n"
        f"보낸 사람: {email['sender']}\n제목: {email['subject']}\n본문:\n{email['body']}"
    )
    result = llm_parse(system, user, Classification)
    print(f"   🔎 분류: {result.route} (긴급도 {result.urgency}) - {result.reason}")
    return {"classification": result.model_dump()}


def route_email(state: EmailState) -> Literal["mark_spam", "auto_reply", "escalate"]:
    return {
        "spam": "mark_spam",
        "simple": "auto_reply",
        "escalate": "escalate",
    }[state["classification"]["route"]]


def mark_spam(state: EmailState) -> dict:
    print("   🗑️  스팸 폴더로 이동")
    return {"outcome": "spam"}


def auto_reply(state: EmailState) -> Command[Literal["escalate", "__end__"]]:
    email = state["email"]
    draft = llm_parse(
        system=(
            "당신은 AnyCompany Consulting 고객 지원 담당자입니다. "
            "FAQ에 있는 정보만 사용해 정중하고 간결한 한국어 답장을 작성하세요. "
            "FAQ로 답할 수 없으면 answered_from_faq=false로 표시하세요.\n"
            f"\nFAQ:\n{FAQ}"
        ),
        user=(
            f"이전 대화:\n{format_history(state.get('history', [])[:-1])}\n\n"
            f"제목: {email['subject']}\n본문:\n{email['body']}"
        ),
        schema=ReplyDraft,
    )
    # 안전장치: 자동 답변에 자신이 없으면 에스컬레이션으로 넘김
    if not draft.answered_from_faq:
        print("   ↪️  FAQ로 답변 불가 → 에스컬레이션")
        return Command(goto="escalate")

    print(f"   ✉️  자동 답장 발송:\n{indent(draft.reply)}")
    return Command(
        goto=END,
        update={
            "outcome": "auto_replied",
            "reply": draft.reply,
            "history": [{"role": "agent(auto)", "content": draft.reply}],
        },
    )


def escalate(state: EmailState) -> dict:
    """티켓을 생성하거나 기존 티켓에 후속 메일을 연결."""
    if state.get("ticket_id"):
        print(f"   🚨 에스컬레이션: 기존 티켓 {state['ticket_id']}에 후속 메일 추가")
        return {}
    ticket_id = f"TCK-{uuid.uuid4().hex[:6].upper()}"
    print(f"   🚨 에스컬레이션: 신규 티켓 {ticket_id} 생성")
    return {"ticket_id": ticket_id}


def human_review(state: EmailState) -> dict:
    """사람 담당자의 답변을 기다린다 (interrupt).

    재개(resume) 시 이 노드는 처음부터 다시 실행되므로, 티켓 생성 같은 부수효과는
    앞 노드(escalate)에 두고 여기에는 interrupt만 둔다.
    """
    cls = state.get("classification", {})
    human_reply = interrupt({
        "ticket_id": state["ticket_id"],
        "urgency": cls.get("urgency"),
        "summary": cls.get("summary"),
        "email": state["email"],
        "history": state.get("history", []),
    })
    return {"reply": human_reply}


def send_human_reply(state: EmailState) -> dict:
    print(f"   👩‍💼 담당자 답장 발송 ({state['ticket_id']}):\n{indent(state['reply'])}")
    return {
        "outcome": "escalated",
        "history": [{"role": f"agent(human, {state['ticket_id']})", "content": state["reply"]}],
    }

## 8. 그래프 구성

- `add_conditional_edges`: `classify` 결과에 따라 경로를 나눕니다 (**라우팅**).
- `checkpointer`: `thread_id`별로 상태를 저장해 후속 메일 추적과 `interrupt` 재개를 가능하게 합니다.

그래프에서 `*` 실선은 고정 경로, `.` 점선은 조건부 경로입니다.

In [ ]:
def build_graph(checkpointer=None):
    builder = StateGraph(EmailState)
    builder.add_node("load_thread", load_thread)
    builder.add_node("classify", classify)
    builder.add_node("mark_spam", mark_spam)
    builder.add_node("auto_reply", auto_reply)
    builder.add_node("escalate", escalate)
    builder.add_node("human_review", human_review)
    builder.add_node("send_human_reply", send_human_reply)

    builder.add_edge(START, "load_thread")
    builder.add_edge("load_thread", "classify")
    builder.add_conditional_edges("classify", route_email)
    builder.add_edge("mark_spam", END)
    builder.add_edge("escalate", "human_review")
    builder.add_edge("human_review", "send_human_reply")
    builder.add_edge("send_human_reply", END)

    # 실서비스: SqliteSaver / PostgresSaver 등으로 바꾸면 재시작 후에도 스레드 상태가 유지됨
    return builder.compile(checkpointer=checkpointer or InMemorySaver())


graph = build_graph()
print(graph.get_graph().draw_ascii())

## 9. 이메일 처리 함수

그래프를 실행하다 `interrupt`로 멈추면(`__interrupt__`), 담당자 답변을 받아 `Command(resume=...)`로 이어서 실행합니다. `human_reply_fn`을 주지 않으면 `input()`으로 직접 입력받습니다.

In [ ]:
def process_email(graph, thread_id: str, email: Email, human_reply_fn=None) -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    result = graph.invoke({"email": email}, config)

    while "__interrupt__" in result:
        payload = result["__interrupt__"][0].value
        if human_reply_fn:
            reply = human_reply_fn(payload)
        else:
            reply = input(f"[{payload['ticket_id']}] 담당자 답변 입력> ")
        result = graph.invoke(Command(resume=reply), config)
    return result


def simulated_agent(payload: dict) -> str:
    """데모용 담당자. 실제로는 상담원 UI, Slack 등에서 답변을 받습니다."""
    return (
        f"안녕하세요, 지원팀 김지원입니다. [{payload['ticket_id']}]\n"
        "문의 주신 건을 최우선으로 확인 중이며, 1시간 내에 진행 상황을 다시 안내드리겠습니다."
    )

## 10. 데모: 샘플 받은편지함 처리

스팸 → 단순 질문 → 긴급 장애 → 같은 스레드의 후속 메일 2통 순서로 처리합니다. 같은 `thread_id`를 쓰면 후속 메일로 인식됩니다.

In [ ]:
inbox = [
    ("thread-001", {
        "sender": "promo@win-prize.biz",
        "subject": "축하합니다! 아이폰 당첨",
        "body": "아래 링크를 클릭하고 카드 정보를 입력하면 경품을 받으실 수 있습니다.",
    }),
    ("thread-002", {
        "sender": "minji@client.co.kr",
        "subject": "비밀번호를 잊어버렸어요",
        "body": "로그인이 안 되는데 비밀번호는 어떻게 재설정하나요?",
    }),
    ("thread-003", {
        "sender": "cto@bigcorp.com",
        "subject": "[긴급] 프로덕션 대시보드 전체 접속 불가",
        "body": "오늘 오전 10시부터 전 직원이 대시보드에 접속하지 못하고 있습니다. 500 에러가 발생합니다.",
    }),
    # thread-003 후속 메일 → 같은 티켓으로 추적
    ("thread-003", {
        "sender": "cto@bigcorp.com",
        "subject": "RE: [긴급] 프로덕션 대시보드 전체 접속 불가",
        "body": "아직도 복구가 안 됐습니다. 예상 복구 시간을 알려주세요.",
    }),
    # thread-002 후속 메일 → 이전 맥락을 참고해 자동 응답
    ("thread-002", {
        "sender": "minji@client.co.kr",
        "subject": "RE: 비밀번호를 잊어버렸어요",
        "body": "감사합니다, 해결됐어요! 혹시 청구서는 어디서 받을 수 있나요?",
    }),
]

for thread_id, email in inbox:
    result = process_email(graph, thread_id, email, simulated_agent)
    print(f"   ✅ 결과: {result['outcome']}")

## 11. 스레드별 누적 상태 확인

checkpointer에 저장된 `thread-003`의 티켓과 대화 이력입니다. 후속 메일이 같은 티켓에 묶여 있습니다.

In [ ]:
state = graph.get_state({"configurable": {"thread_id": "thread-003"}}).values
print(f"ticket_id = {state['ticket_id']}")
for h in state["history"]:
    print(f" - [{h['role']}] {h['content'][:70]}")

## 12. (선택) 담당자로서 직접 답변해 보기

`human_reply_fn` 없이 실행하면 에스컬레이션될 때 입력창이 뜹니다. 입력한 내용이 고객에게 답장으로 나갑니다.

In [ ]:
# 주석을 풀고 실행하세요.
# process_email(graph, "thread-004", {
#     "sender": "legal@partner.com",
#     "subject": "계약 해지 및 환불 요청",
#     "body": "지난달 컨설팅 결과물이 계약 내용과 달라 계약 해지와 전액 환불을 요청합니다.",
# })

## 13. 운영 환경으로 확장할 때

- **상태 영속화:** `InMemorySaver` 대신 `langgraph-checkpoint-postgres`의 `PostgresSaver`(예: Amazon Aurora PostgreSQL) 같은 저장소를 쓰면 재시작 후에도 스레드와 대기 중인 에스컬레이션이 유지됩니다.
- **지식 베이스:** FAQ 문자열을 Amazon Bedrock Knowledge Bases 검색 결과로 바꿉니다.
- **대량 처리:** Amazon SES로 메일을 받아 Amazon SQS에 넣고, AWS Lambda나 Amazon ECS 워커가 `process_email`을 실행하는 구조로 시간당 1,000통 이상을 처리할 수 있습니다. `thread_id`는 메일의 `Message-ID`/`In-Reply-To` 헤더로 만듭니다.
- **비용 최적화:** 스팸 분류는 Nova Micro, 답장 작성은 Nova 2 Lite처럼 노드마다 다른 모델을 쓸 수 있습니다.
- **가드레일:** Converse 호출에 `guardrailConfig`로 Amazon Bedrock Guardrails를 연결하면 자동 답장에서 개인정보·부적절한 내용을 걸러낼 수 있습니다.